In [1]:
!nvidia-smi

Sun Aug 11 00:41:23 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   66C    P8              11W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
!pip install ortools
!pip install pycuda
!pip install pyopencl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.3/309.3 kB 25.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires protobuf<5,>=3.20, but you have protobuf 5.27.3 which is incompatible.
google-ai-generativelanguage 0.6.6 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.2

In [3]:
from ortools.algorithms.python import knapsack_solver
import pycuda.autoinit
import pycuda.driver as cuda
import numpy as np
from pycuda.compiler import SourceModule
import time
import threading

In [ ]:
# Colab starts with an empty filesystem, so pull the project in to get problems.py.
import os
import sys

REPO_URL = "https://github.com/andrewrowell/knapsack-gpu.git"
REPO_DIR = "/content/knapsack-gpu"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
NUMBER_OF_TRIALS = 100
PREVIEW_ITEMS = 15

import numpy as np

import problems

# Every implementation in this project solves the identical set of problems by
# reading them from problems.py.
problem_set = problems.generate()

num_problems = problem_set.num_problems
num_items_per_problem = problem_set.num_items
max_capacity = problem_set.max_capacity
capacities = problem_set.capacities

# Subset sum: an item's value is its weight, so the solver cells below are
# handed the same array for both.
values = weights = problem_set.items

num_items = np.full(num_problems, num_items_per_problem, dtype=np.int32)
max_values = np.zeros(num_problems, dtype=np.int32)

print(f"{num_problems} problems, {num_items_per_problem} items each, capacity at most {max_capacity}")
for i in range(3):
    row = problem_set.items[i]
    head = ", ".join(str(item) for item in row[:PREVIEW_ITEMS])
    rest = f", ... ({len(row) - PREVIEW_ITEMS} more)" if len(row) > PREVIEW_ITEMS else ""
    print(f"Problem {i + 1}: capacity {problem_set.capacities[i]}, items [{head}{rest}]")

In [5]:
# CUDA kernel to solve multiple knapsack problems
kernel_code = """
__global__ void knapsack(int *values, int *weights, int *capacities, int *num_items, int *max_values, int max_capacity) {
    int problem_idx = blockIdx.x;
    int item_idx = threadIdx.x;

    extern __shared__ int dp[];

    // Initialize DP table
    for (int w = item_idx; w <= max_capacity; w += blockDim.x) {
        dp[w] = 0;
    }
    __syncthreads();

    // Populate DP table
    for (int i = 0; i < num_items[problem_idx]; i++) {
        int weight = weights[problem_idx * num_items[problem_idx] + i];
        int value = values[problem_idx * num_items[problem_idx] + i];

        for (int w = max_capacity; w >= weight; w--) {
            atomicMax(&dp[w], dp[w - weight] + value);
        }
        __syncthreads();
    }

    // Write result
    if (item_idx == 0) {
        max_values[problem_idx] = dp[capacities[problem_idx]];
    }
}
"""

# Compile the kernel code
mod = SourceModule(kernel_code)

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Host data is already constructed
    max_values = np.zeros(num_problems, dtype=np.int32)

    # Measure execution time
    start_time = time.time()

    # Allocate memory on the GPU
    values_gpu = cuda.mem_alloc(values.nbytes)
    weights_gpu = cuda.mem_alloc(weights.nbytes)
    capacities_gpu = cuda.mem_alloc(capacities.nbytes)
    num_items_gpu = cuda.mem_alloc(num_items.nbytes)
    max_values_gpu = cuda.mem_alloc(max_values.nbytes)

    # Copy data to the GPU
    cuda.memcpy_htod(values_gpu, values)
    cuda.memcpy_htod(weights_gpu, weights)
    cuda.memcpy_htod(capacities_gpu, capacities)
    cuda.memcpy_htod(num_items_gpu, num_items)
    cuda.memcpy_htod(max_values_gpu, max_values)

    # Launch the kernel
    knapsack = mod.get_function("knapsack")
    shared_memory_size = (max_capacity + 1) * 4  # size of shared memory for one problem
    knapsack(values_gpu, weights_gpu, capacities_gpu, num_items_gpu, max_values_gpu, np.int32(max_capacity),
            block=(1024, 1, 1), grid=(num_problems, 1), shared=shared_memory_size)

    # Copy the result back to the CPU
    cuda.memcpy_dtoh(max_values, max_values_gpu)

    # Print execution time
    end_time = time.time()
    #print(f"CUDA execution time: {end_time - start_time:.6f} seconds")
    execution_times.append(end_time - start_time)

print(f"Average PyCUDA execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Print the results
#for i in range(num_problems):
for i in range(10):
    print(f"Problem {i + 1}: Maximum value = {max_values[i]}")

/usr/local/lib/python3.10/dist-packages/google/colab/_variable_inspector.py:27: UserWarning: device_allocation in out-of-thread context could not be cleaned up
  globals().clear()
/usr/local/lib/python3.10/dist-packages/google/colab/_variable_inspector.py:27: UserWarning: device_allocation in out-of-thread context could not be cleaned up
  globals().clear()


Average PyCUDA execution time: 0.007630 seconds
Problem 1: Maximum value = 145
Problem 2: Maximum value = 61
Problem 3: Maximum value = 336
Problem 4: Maximum value = 88
Problem 5: Maximum value = 53
Problem 6: Maximum value = 0
Problem 7: Maximum value = 38
Problem 8: Maximum value = 102
Problem 9: Maximum value = 156
Problem 10: Maximum value = 49


/usr/local/lib/python3.10/dist-packages/google/colab/_variable_inspector.py:27: UserWarning: device_allocation in out-of-thread context could not be cleaned up
  globals().clear()


In [6]:
# OR-Tools Section

# This section should run in Colab T4
import platform
print(platform.node())

# Function to solve a single knapsack problem using OR-Tools
def solve_knapsack(values, weights, capacity, problem_idx, results):
    solver = knapsack_solver.KnapsackSolver(
        knapsack_solver.KNAPSACK_MULTIDIMENSION_BRANCH_AND_BOUND_SOLVER, 'KnapsackExample')

    solver.init(values, [weights], [capacity])

    max_value = solver.solve()
    results[problem_idx] = max_value
    #print(f"Problem {problem_idx + 1}: Maximum value = {max_value}")

# Storage for results
results = [0] * num_problems

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    # Create and start threads
    threads = []
    for i in range(num_problems):
        t = threading.Thread(target=solve_knapsack, args=(values[i], weights[i], capacities[i], i, results))
        threads.append(t)
        t.start()

    # Wait for all threads to complete
    for t in threads:
        t.join()

    # Print execution time
    end_time = time.time()
    #print(f"Threaded execution time: {end_time - start_time:.6f} seconds")
    execution_times.append(end_time - start_time)

print(f"Average CPU execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Print the final results
#for i in range(num_problems):
for i in range(10):
    print(f"Final Result for Problem {i + 1}: Maximum value = {results[i]}")


/usr/local/lib/python3.10/dist-packages/google/colab/_variable_inspector.py:27: UserWarning: device_allocation in out-of-thread context could not be cleaned up
  globals().clear()


0660d59b22b6
Average CPU execution time: 1.374059 seconds
Final Result for Problem 1: Maximum value = 145
Final Result for Problem 2: Maximum value = 61
Final Result for Problem 3: Maximum value = 162
Final Result for Problem 4: Maximum value = 88
Final Result for Problem 5: Maximum value = 53
Final Result for Problem 6: Maximum value = 0
Final Result for Problem 7: Maximum value = 38
Final Result for Problem 8: Maximum value = 81
Final Result for Problem 9: Maximum value = 156
Final Result for Problem 10: Maximum value = 49


In [7]:
!sudo apt update
!sudo apt purge *nvidia* -y
!sudo apt install nvidia-driver-530 -y

Streaming output truncated to the last 5000 lines.
Package 'linux-objects-nvidia-525-6.2.0-1021-gcp' is not installed, so not removed
Package 'linux-objects-nvidia-525-6.2.0-25-generic' is not installed, so not removed
Package 'linux-objects-nvidia-525-6.2.0-26-generic' is not installed, so not removed
Package 'linux-objects-nvidia-525-6.2.0-31-generic' is not installed, so not removed
Package 'linux-objects-nvidia-525-6.2.0-32-generic' is not installed, so not removed
Package 'linux-objects-nvidia-525-6.2.0-33-generic' is not installed, so not removed
Package 'linux-objects-nvidia-525-6.2.0-34-generic' is not installed, so not removed
Package 'linux-objects-nvidia-525-6.2.0-35-generic' is not installed, so not removed
Package 'linux-objects-nvidia-525-6.2.0-36-generic' is not installed, so not removed
Package 'linux-objects-nvidia-525-6.2.0-37-generic' is not installed, so not removed
Package 'linux-objects-nvidia-525-6.2.0-39-generic' is not installed, so not removed
Package 'linux-o

In [14]:
import numpy as np
import pyopencl as cl

print(cl.get_platforms())

# OpenCL setup
platform = cl.get_platforms()[0]
device = platform.get_devices()[0]
context = cl.Context([device])
queue = cl.CommandQueue(context)

# Kernel code for solving the knapsack problem
knapsack_kernel = """
__kernel void solve_knapsack(
    __global const int *values,
    __global const int *weights,
    __global const int *capacities,
    __global int *results,
    const int num_items,
    const int num_problems
) {
    int problem_idx = get_global_id(0);
    if (problem_idx < num_problems) {
        int capacity = capacities[problem_idx];
        int dp[51] = {0};  // Fixed size for the DP table, considering max_capacity = 50

        // Fill DP table using dynamic programming
        for (int i = 0; i < num_items; i++) {
            int value = values[problem_idx * num_items + i];
            int weight = weights[problem_idx * num_items + i];
            for (int j = capacity; j >= weight; j--) {
                int new_val = dp[j - weight] + value;
                if (new_val > dp[j]) {
                    dp[j] = new_val;
                }
            }
        }

        // Write the result to the results array
        results[problem_idx] = dp[capacity];
    }
}
"""

# Compile the kernel code
program = cl.Program(context, knapsack_kernel).build()

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Create buffers
    values_buf = cl.Buffer(context, cl.mem_flags.READ_ONLY | cl.mem_flags.COPY_HOST_PTR, hostbuf=values.flatten())
    weights_buf = cl.Buffer(context, cl.mem_flags.READ_ONLY | cl.mem_flags.COPY_HOST_PTR, hostbuf=weights.flatten())
    capacities_buf = cl.Buffer(context, cl.mem_flags.READ_ONLY | cl.mem_flags.COPY_HOST_PTR, hostbuf=capacities)
    results_buf = cl.Buffer(context, cl.mem_flags.WRITE_ONLY, size=4 * num_problems)

    # Execute the kernel
    global_size = (num_problems,)
    local_size = None  # Let OpenCL determine the local size

    start_time = time.time()

    program.solve_knapsack(queue, global_size, local_size,
                          values_buf, weights_buf, capacities_buf,
                          results_buf, np.int32(num_items_per_problem), np.int32(num_problems))

    # Retrieve results
    results = np.zeros(num_problems, dtype=np.int32)
    cl.enqueue_copy(queue, results, results_buf).wait()

    # Measure execution time
    end_time = time.time()
    #print(f"OpenCL execution time: {end_time - start_time:.6f} seconds")
    execution_times.append(end_time - start_time)

print(f"Average CPU execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Print the final results
print("First 10 results:")
for i in range(10):
    print(f"Final Result for Problem {i + 1}: Maximum value = {results[i]}")

[<pyopencl.Platform 'NVIDIA CUDA' at 0x5bf263cd45a0>]
Average CPU execution time: 0.000615 seconds
First 10 results:
Final Result for Problem 1: Maximum value = 145
Final Result for Problem 2: Maximum value = 61
Final Result for Problem 3: Maximum value = 162
Final Result for Problem 4: Maximum value = 88
Final Result for Problem 5: Maximum value = 53
Final Result for Problem 6: Maximum value = 0
Final Result for Problem 7: Maximum value = 38
Final Result for Problem 8: Maximum value = 81
Final Result for Problem 9: Maximum value = 156
Final Result for Problem 10: Maximum value = 49


In [9]:
# This cell should be run on my macbook
print(platform.node())

# Storage for results
results = [0] * num_problems

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    # Create and start threads
    threads = []
    for i in range(num_problems):
        t = threading.Thread(target=solve_knapsack, args=(values[i], weights[i], capacities[i], i, results))
        threads.append(t)
        t.start()

    # Wait for all threads to complete
    for t in threads:
        t.join()

    # Print execution time
    end_time = time.time()
    #print(f"Threaded execution time: {end_time - start_time:.6f} seconds")
    execution_times.append(end_time - start_time)

print(f"Average CPU execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Print the final results
#for i in range(num_problems):
for i in range(10):
    print(f"Final Result for Problem {i + 1}: Maximum value = {results[i]}")

AttributeError: 'Platform' object has no attribute 'node'

In [ ]:
# This cell should be run on my macbook
print(platform.node())

print(cl.get_platforms())

# OpenCL setup
platform = cl.get_platforms()[0]
device = platform.get_devices()[0]
context = cl.Context([device])
queue = cl.CommandQueue(context)

# Kernel code for solving the knapsack problem
knapsack_kernel = """
__kernel void solve_knapsack(
    __global const int *values,
    __global const int *weights,
    __global const int *capacities,
    __global int *results,
    const int num_items,
    const int num_problems
) {
    int problem_idx = get_global_id(0);
    if (problem_idx < num_problems) {
        int capacity = capacities[problem_idx];
        int dp[51] = {0};  // Fixed size for the DP table, considering max_capacity = 50

        // Fill DP table using dynamic programming
        for (int i = 0; i < num_items; i++) {
            int value = values[problem_idx * num_items + i];
            int weight = weights[problem_idx * num_items + i];
            for (int j = capacity; j >= weight; j--) {
                int new_val = dp[j - weight] + value;
                if (new_val > dp[j]) {
                    dp[j] = new_val;
                }
            }
        }

        // Write the result to the results array
        results[problem_idx] = dp[capacity];
    }
}
"""

# Compile the kernel code
program = cl.Program(context, knapsack_kernel).build()

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Create buffers
    values_buf = cl.Buffer(context, cl.mem_flags.READ_ONLY | cl.mem_flags.COPY_HOST_PTR, hostbuf=values.flatten())
    weights_buf = cl.Buffer(context, cl.mem_flags.READ_ONLY | cl.mem_flags.COPY_HOST_PTR, hostbuf=weights.flatten())
    capacities_buf = cl.Buffer(context, cl.mem_flags.READ_ONLY | cl.mem_flags.COPY_HOST_PTR, hostbuf=capacities)
    results_buf = cl.Buffer(context, cl.mem_flags.WRITE_ONLY, size=4 * num_problems)

    # Execute the kernel
    global_size = (num_problems,)
    local_size = None  # Let OpenCL determine the local size

    start_time = time.time()

    program.solve_knapsack(queue, global_size, local_size,
                          values_buf, weights_buf, capacities_buf,
                          results_buf, np.int32(num_items_per_problem), np.int32(num_problems))

    # Retrieve results
    results = np.zeros(num_problems, dtype=np.int32)
    cl.enqueue_copy(queue, results, results_buf).wait()

    # Measure execution time
    end_time = time.time()
    #print(f"OpenCL execution time: {end_time - start_time:.6f} seconds")
    execution_times.append(end_time - start_time)

print(f"Average CPU execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Print the final results
print("First 10 results:")
for i in range(10):
    print(f"Final Result for Problem {i + 1}: Maximum value = {results[i]}")